# 03 - Symptom-disorder comorbidity graph (novelty N2)

Builds the heterogeneous graph that lets evidence propagate between related
symptoms and comorbid disorders, instead of treating each condition as
independent. Nodes are the 22 unified symptoms plus 14 disorders, with three
relation types:

| Relation | Source | Role |
|---|---|---|
| `symptom -> disorder` | DSM-5-informed prior (`configs/dsm5_prior.json`) | the `A0` term in `A = A0 + dA` |
| `symptom <-> symptom` | PPMI over co-annotated gold labels | which symptoms genuinely co-occur |
| `disorder <-> disorder` | authors posting in several subreddits | an empirical comorbidity proxy |

**Everything empirical uses the training split only**, so no test information
reaches the graph prior.

Runs on CPU in a couple of minutes. No GPU needed.

## 1. Mount Drive and get the code

In [ ]:
import os, sys
from google.colab import drive

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
DRIVE_PROCESSED = f'{DRIVE_BASE}/processed'
DRIVE_GRAPH = f'{DRIVE_BASE}/graph'
LOCAL_PROCESSED = '/content/processed'
LOCAL_GRAPH = '/content/graph'

os.environ.update(DRIVE_PROCESSED=DRIVE_PROCESSED, DRIVE_GRAPH=DRIVE_GRAPH,
                  LOCAL_PROCESSED=LOCAL_PROCESSED, LOCAL_GRAPH=LOCAL_GRAPH)
assert os.path.isdir(DRIVE_PROCESSED), (
    f'{DRIVE_PROCESSED} not found - run notebook 02 first')
print(sorted(os.listdir(DRIVE_PROCESSED)))

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q --branch main $REPO $CHECKOUT

need = ['src/symptograph/graph.py', 'configs/dsm5_prior.json']
missing = [f for f in need if not os.path.isfile(f'{CHECKOUT}/{f}')]
assert not missing, f'stale checkout, missing {missing} - run: !rm -rf {CHECKOUT}'
sys.path.insert(0, f'{CHECKOUT}/src')
!git -C $CHECKOUT log --oneline -1

Node features are sentence embeddings of each symptom definition, which give the
graph a meaningful starting representation instead of random vectors.

In [ ]:
!pip install -q sentence-transformers 2>&1 | tail -1
!mkdir -p $LOCAL_PROCESSED && rsync -ah "$DRIVE_PROCESSED/" "$LOCAL_PROCESSED/"
print(sorted(os.listdir(LOCAL_PROCESSED)))

## 2. Build the graph

In [ ]:
import numpy as np, pandas as pd, json
from symptograph import graph as G

g, paths, stats = G.build(LOCAL_PROCESSED, LOCAL_GRAPH,
                          min_support=10, with_features=True, verbose=True)
print()
print(json.dumps({k: v for k, v in stats.items()
                  if k not in ('prior', 'disorder_cooccurrence')}, indent=2))

## 3. Symptom-symptom co-occurrence

### An honest limitation worth reporting

Only **31 of the 231 symptom pairs are observable**. No open corpus annotates
the full symptom set jointly: DepressionEmo co-annotates 8 symptoms, IRF 2 and
LoST 1, so most cells have no data behind them at all. The `A_ss_mask` records
exactly which cells are supported, and the rest must come from the DSM-5 prior
and from weak labels (notebook 04). Reporting this is better than quietly
emitting zeros that look like measured independence.

In [ ]:
print(f"observable pairs: {stats['symptom_pairs_observable']} of "
      f"{stats['symptom_pairs_total']}")
print(f"pairs with positive PPMI: {stats['symptom_edges_ppmi']}\n")
display(G.top_edges(g.A_ss, g.symptoms, g.symptoms, k=15,
                    support=g.A_ss_support, symmetric=True))

## 4. Disorder-disorder comorbidity

### Why PMI alone does not work here

Every author in the table has at least one subreddit membership by construction,
and authors average only **1.125 disorders**. Observed co-membership therefore
falls *below* the independence expectation for every pair (maximum lift 0.99),
so plain PPMI would zero out the whole matrix. Two measures are kept instead:

- **`A_dd` = p(b | a)** - conditional co-membership, used as the edge weight.
  Dense, non-negative and directly meaningful for message passing.
- **`A_dd_ppmi`** - enrichment computed only over authors with two or more
  disorders, where co-membership is the norm and the null is sensible.

The enrichment view is a useful sanity check: it recovers comorbidities that are
well established clinically, which is reassuring given the labels are only
subreddit memberships.

In [ ]:
print(json.dumps(stats['disorder_cooccurrence'], indent=2))
print('\nStrongest conditional co-membership, p(b | a):')
display(G.top_edges(g.A_dd, g.disorders, g.disorders, k=10,
                    support=g.A_dd_support, symmetric=False))
print('Most enriched comorbidities (PPMI over multi-disorder authors):')
display(G.top_edges(g.A_dd_ppmi, g.disorders, g.disorders, k=10,
                    support=g.A_dd_support, symmetric=True))

## 5. Does the data support the hand-authored prior?

The prior claims certain symptoms characterise a disorder, which implies those
symptoms should co-occur in posts. This checks the observable subset of each
disorder's implied pairs. A high support rate means the prior is consistent with
the data wherever it can be checked; `NaN` means nothing was observable.

The prior is **authored by this project** from DSM-5 criterion sets and the
PHQ-9/GAD-7 item content, not taken from an existing resource, so it should be
reviewed by a clinician before publication.

In [ ]:
print(stats['prior']['provenance'], '\n')
print('disorders deliberately without a prior:',
      stats['prior']['disorders_without_prior'], '\n')
display(G.prior_support_report(g))

## 6. Visualise the three relation types

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(19, 6))

ax = axes[0]
im = ax.imshow(g.A_sd, cmap='Blues', aspect='auto')
ax.set_title('symptom -> disorder (DSM-5 prior)')
ax.set_xticks(range(len(g.disorders)))
ax.set_xticklabels(g.disorders, rotation=90, fontsize=8)
ax.set_yticks(range(len(g.symptoms)))
ax.set_yticklabels(g.symptoms, fontsize=8)

ax = axes[1]
masked = np.where(g.A_ss_mask > 0, g.A_ss, np.nan)
im = ax.imshow(masked, cmap='viridis')
ax.set_title('symptom <-> symptom PPMI\n(grey = not observable)')
ax.set_xticks(range(len(g.symptoms)))
ax.set_xticklabels(g.symptoms, rotation=90, fontsize=7)
ax.set_yticks(range(len(g.symptoms)))
ax.set_yticklabels(g.symptoms, fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046)

ax = axes[2]
im = ax.imshow(g.A_dd_ppmi, cmap='magma')
ax.set_title('disorder <-> disorder enrichment')
ax.set_xticks(range(len(g.disorders)))
ax.set_xticklabels(g.disorders, rotation=90, fontsize=8)
ax.set_yticks(range(len(g.disorders)))
ax.set_yticklabels(g.disorders, fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(f'{LOCAL_GRAPH}/graph_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save to Drive

In [ ]:
!mkdir -p "$DRIVE_GRAPH"
!rsync -ah --info=progress2 "$LOCAL_GRAPH/" "$DRIVE_GRAPH/"
!ls -lh "$DRIVE_GRAPH"

reloaded = G.SymptomGraph.load(DRIVE_GRAPH)
print('\nreload check')
print('  symptoms       ', len(reloaded.symptoms))
print('  disorders      ', len(reloaded.disorders))
print('  node features  ', None if reloaded.node_features is None
                             else reloaded.node_features.shape)
print('  feature model  ', reloaded.feature_model)

## Notes

`graph.npz` holds six matrices: `A_sd` (prior), `A_ss` with `A_ss_mask` and
`A_ss_support`, and `A_dd` with `A_dd_ppmi` and `A_dd_support`. The model should
treat these as the fixed `A0` and learn a sparse correction `dA` on top, with an
L1 penalty, so new edges the data supports can be inspected and reported.

Use `A_ss_mask` to avoid penalising the model over unobservable cells.

**Caveat to state in the paper.** The disorder edges come from subreddit
co-membership by pseudonymised authors, not from clinical diagnoses. It is a
proxy, and a reasonable one given the enrichment recovers established
comorbidities, but it is not diagnostic ground truth.

**Next:** notebook 04 weak-labels the 11 symptoms with no gold data.